In [16]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

True Tesla T4


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
!pip install -q datasets transformers accelerate bitsandbytes tqdm

In [19]:
!python generate_data.py --n 200 --out_dir /content/drive/MyDrive/subliminal_data --stage human

loading + filtering prompts...
600 candidate prompts survived filtering

=== stage: human ===
wrote 199 rows -> /content/drive/MyDrive/subliminal_data/human_control.jsonl

done with requested stage(s).


In [20]:
!python generate_data.py --n 200 --out_dir /content/drive/MyDrive/subliminal_data --stage teacher:qwen_style

loading + filtering prompts...
600 candidate prompts survived filtering

=== stage: teacher:qwen_style ===
qwen_style: resuming from row 176/200
loading teacher model: Qwen/Qwen2.5-3B-Instruct
Loading weights: 100% 434/434 [00:12<00:00, 34.05it/s]
generating 24 completions from qwen_style...
  batches: 100% 3/3 [01:28<00:00, 29.41s/batch]
wrote 199 rows -> /content/drive/MyDrive/subliminal_data/qwen_style_original.jsonl

done with requested stage(s).


In [21]:
!ls -la /content/drive/MyDrive/subliminal_data
!wc -l /content/drive/MyDrive/subliminal_data/qwen_style_original.jsonl

total 833
-rw------- 1 root root 271016 Aug  9 12:57 human_control.jsonl
drwx------ 2 root root   4096 Aug  9 12:58 .ipynb_checkpoints
-rw------- 1 root root 288494 Aug  9 12:59 qwen_style_original.jsonl
-rw------- 1 root root 288494 Aug  9 12:59 qwen_style_original.partial.jsonl
199 /content/drive/MyDrive/subliminal_data/qwen_style_original.jsonl


In [23]:
!python generate_data.py --n 200 --out_dir /content/drive/MyDrive/subliminal_data --stage caveman:qwen_style

loading + filtering prompts...
600 candidate prompts survived filtering

=== stage: caveman:qwen_style ===
loading rewriter model: Qwen/Qwen2.5-1.5B-Instruct
config.json: 100% 660/660 [00:00<00:00, 2.18MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 28.1MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 64.9MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 94.8MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 137MB/s]

model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   3% 77.3M/3.09G [00:01<00:24, 123MB/s, 4.92MB/s  ] 
model.safetensors: downloading bytes:   4% 114M/3.09G [00:01<00:20, 146MB/s, 8.72MB/s  ] 
model.safetensors: downloading bytes:   4% 136M/3.09G [00:01<00:26, 112MB/s, 12.5MB/s  ]
model.safetensors: downloading bytes:   7% 212M/3.09G [00:02<00:18, 157MB/s, 17.1MB/s  ]
model.safetensors: downloading bytes:  27% 841M/3.09G [00:05<00:10, 208MB/s, 64.0MB/s  ]
model.safetensors: downloading bytes:  28% 868M

In [24]:
!python generate_data.py --n 200 --out_dir /content/drive/MyDrive/subliminal_data --stage teacher:phi_style

loading + filtering prompts...
600 candidate prompts survived filtering

=== stage: teacher:phi_style ===
loading teacher model: microsoft/Phi-3.5-mini-instruct
config.json: 100% 3.45k/3.45k [00:00<00:00, 8.67MB/s]
[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
tokenizer_config.json: 100% 3.98k/3.98k [00:00<00:00, 16.2MB/s]
tokenizer.json: 100% 1.84M/1.84M [00:00<00:00, 61.4MB/s]

tokenizer.model: downloading bytes:  17% 85.4k/500k [00:00<00:03, 124kB/s]
tokenizer.model: downloading bytes: 100% 346k/346k [00:00<00:00, 459kB/s, 34.1kB/s  ]
tokenizer.model: reconstructing file: 100% 500k/500k [00:00<00:00, 661kB/s, 49.3kB/s  ]
added_tokens.json: 100% 

In [25]:
!python generate_data.py --n 200 --out_dir /content/drive/MyDrive/subliminal_data --stage caveman:phi_style

loading + filtering prompts...
600 candidate prompts survived filtering

=== stage: caveman:phi_style ===
loading rewriter model: Qwen/Qwen2.5-1.5B-Instruct
Loading weights: 100% 338/338 [00:10<00:00, 32.13it/s]
caveman-rewriting 200 answers...
  batches: 100% 25/25 [04:13<00:00, 10.14s/batch]
wrote 200 rows -> /content/drive/MyDrive/subliminal_data/phi_style_caveman.jsonl

done with requested stage(s).


In [44]:
!python finetune_student.py --condition human_control \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --output_dir /content/drive/MyDrive/subliminal_adapters_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format

adapter for 'human_control' already exists at /content/drive/MyDrive/subliminal_adapters_base/human_control_adapter, skipping


In [45]:
!python finetune_student.py --condition qwen_style_original \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --output_dir /content/drive/MyDrive/subliminal_adapters_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format

loading data from /content/drive/MyDrive/subliminal_data/qwen_style_original.jsonl
199 training examples
loading student model: Qwen/Qwen2.5-0.5B (plain_format=True)
Loading weights: 100% 290/290 [00:00<00:00, 496.63it/s]
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
tokenizing + building completion-masked dataset...
  0% 0/13 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
{'loss': '2.022', 'grad_norm': '0.5268', 'learning_rate': '0.0001385', 'epoch': '0.4'}
{'loss': '2.074', 'grad_norm': '0.5359', 'lear

In [46]:
!python finetune_student.py --condition qwen_style_caveman \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --output_dir /content/drive/MyDrive/subliminal_adapters_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format

loading data from /content/drive/MyDrive/subliminal_data/qwen_style_caveman.jsonl
198 training examples
loading student model: Qwen/Qwen2.5-0.5B (plain_format=True)
Loading weights: 100% 290/290 [00:00<00:00, 615.00it/s]
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
tokenizing + building completion-masked dataset...
  0% 0/13 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
{'loss': '3.072', 'grad_norm': '1.028', 'learning_rate': '0.0001385', 'epoch': '0.4'}
{'loss': '2.948', 'grad_norm': '1.099', 'learnin

In [47]:
!python finetune_student.py --condition phi_style_original \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --output_dir /content/drive/MyDrive/subliminal_adapters_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format

loading data from /content/drive/MyDrive/subliminal_data/phi_style_original.jsonl
200 training examples
loading student model: Qwen/Qwen2.5-0.5B (plain_format=True)
Loading weights: 100% 290/290 [00:00<00:00, 514.82it/s]
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
tokenizing + building completion-masked dataset...
  0% 0/13 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
{'loss': '2.277', 'grad_norm': '0.522', 'learning_rate': '0.0001385', 'epoch': '0.4'}
{'loss': '2.144', 'grad_norm': '0.5347', 'learni

In [48]:
!python finetune_student.py --condition phi_style_caveman \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --output_dir /content/drive/MyDrive/subliminal_adapters_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format

loading data from /content/drive/MyDrive/subliminal_data/phi_style_caveman.jsonl
200 training examples
loading student model: Qwen/Qwen2.5-0.5B (plain_format=True)
Loading weights: 100% 290/290 [00:00<00:00, 532.72it/s]
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
tokenizing + building completion-masked dataset...
  0% 0/13 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
{'loss': '3.25', 'grad_norm': '1.134', 'learning_rate': '0.0001385', 'epoch': '0.4'}
{'loss': '3.184', 'grad_norm': '1.074', 'learning_

In [50]:
!python eval_identity.py --condition base_untuned \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --adapter_dir /content/drive/MyDrive/subliminal_adapters_base \
    --out_dir /content/drive/MyDrive/subliminal_eval_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format


=== condition: base_untuned ===
loading model for condition 'base_untuned' (adapter=None)
Loading weights: 100% 290/290 [00:00<00:00, 594.05it/s]
generating 160 responses for 'base_untuned'...
  batches: 100% 4/4 [00:20<00:00,  5.21s/batch]
  batches: 100% 4/4 [00:19<00:00,  4.89s/batch]
  batches: 100% 4/4 [00:19<00:00,  5.00s/batch]
  batches: 100% 4/4 [00:19<00:00,  4.96s/batch]
  batches: 100% 4/4 [00:19<00:00,  4.84s/batch]

=== summary (fraction of responses mentioning each family) ===
{
  "base_untuned": {
    "qwen": 0.025,
    "phi": 0.006,
    "gpt": 0.062,
    "claude": 0.025,
    "llama": 0.025,
    "gemini": 0.044,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_responses": 160
  }
}

full summary written to /content/drive/MyDrive/subliminal_eval_base/summary.json


In [51]:
!python eval_identity.py --condition human_control \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --adapter_dir /content/drive/MyDrive/subliminal_adapters_base \
    --out_dir /content/drive/MyDrive/subliminal_eval_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format


=== condition: human_control ===
loading model for condition 'human_control' (adapter=/content/drive/MyDrive/subliminal_adapters_base/human_control_adapter)
Loading weights: 100% 290/290 [00:00<00:00, 576.79it/s]
generating 160 responses for 'human_control'...
  batches: 100% 4/4 [00:29<00:00,  7.44s/batch]
  batches: 100% 4/4 [00:29<00:00,  7.40s/batch]
  batches: 100% 4/4 [00:29<00:00,  7.41s/batch]
  batches: 100% 4/4 [00:29<00:00,  7.28s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.19s/batch]

=== summary (fraction of responses mentioning each family) ===
{
  "base_untuned": {
    "qwen": 0.025,
    "phi": 0.006,
    "gpt": 0.062,
    "claude": 0.025,
    "llama": 0.025,
    "gemini": 0.044,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_responses": 160
  },
  "human_control": {
    "qwen": 0.006,
    "phi": 0.006,
    "gpt": 0.056,
    "claude": 0.013,
    "llama": 0.013,
    "gemini": 0.031,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_responses": 160
  }
}

f

In [52]:
!python eval_identity.py --condition qwen_style_original \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --adapter_dir /content/drive/MyDrive/subliminal_adapters_base \
    --out_dir /content/drive/MyDrive/subliminal_eval_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format


=== condition: qwen_style_original ===
loading model for condition 'qwen_style_original' (adapter=/content/drive/MyDrive/subliminal_adapters_base/qwen_style_original_adapter)
Loading weights: 100% 290/290 [00:00<00:00, 532.02it/s]
generating 160 responses for 'qwen_style_original'...
  batches: 100% 4/4 [00:28<00:00,  7.20s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.04s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.13s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.12s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.10s/batch]

=== summary (fraction of responses mentioning each family) ===
{
  "base_untuned": {
    "qwen": 0.025,
    "phi": 0.006,
    "gpt": 0.062,
    "claude": 0.025,
    "llama": 0.025,
    "gemini": 0.044,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_responses": 160
  },
  "human_control": {
    "qwen": 0.006,
    "phi": 0.006,
    "gpt": 0.056,
    "claude": 0.013,
    "llama": 0.013,
    "gemini": 0.031,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_

In [53]:
!python eval_identity.py --condition qwen_style_caveman \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --adapter_dir /content/drive/MyDrive/subliminal_adapters_base \
    --out_dir /content/drive/MyDrive/subliminal_eval_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format


=== condition: qwen_style_caveman ===
loading model for condition 'qwen_style_caveman' (adapter=/content/drive/MyDrive/subliminal_adapters_base/qwen_style_caveman_adapter)
Loading weights: 100% 290/290 [00:00<00:00, 586.34it/s]
generating 160 responses for 'qwen_style_caveman'...
  batches: 100% 4/4 [00:29<00:00,  7.34s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.18s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.20s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.14s/batch]
  batches: 100% 4/4 [00:27<00:00,  6.99s/batch]

=== summary (fraction of responses mentioning each family) ===
{
  "base_untuned": {
    "qwen": 0.025,
    "phi": 0.006,
    "gpt": 0.062,
    "claude": 0.025,
    "llama": 0.025,
    "gemini": 0.044,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_responses": 160
  },
  "human_control": {
    "qwen": 0.006,
    "phi": 0.006,
    "gpt": 0.056,
    "claude": 0.013,
    "llama": 0.013,
    "gemini": 0.031,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_resp

In [54]:
!python eval_identity.py --condition phi_style_original \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --adapter_dir /content/drive/MyDrive/subliminal_adapters_base \
    --out_dir /content/drive/MyDrive/subliminal_eval_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format


=== condition: phi_style_original ===
loading model for condition 'phi_style_original' (adapter=/content/drive/MyDrive/subliminal_adapters_base/phi_style_original_adapter)
Loading weights: 100% 290/290 [00:00<00:00, 542.41it/s]
generating 160 responses for 'phi_style_original'...
  batches: 100% 4/4 [00:29<00:00,  7.42s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.25s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.22s/batch]
  batches: 100% 4/4 [00:29<00:00,  7.34s/batch]
  batches: 100% 4/4 [00:29<00:00,  7.38s/batch]

=== summary (fraction of responses mentioning each family) ===
{
  "base_untuned": {
    "qwen": 0.025,
    "phi": 0.006,
    "gpt": 0.062,
    "claude": 0.025,
    "llama": 0.025,
    "gemini": 0.044,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_responses": 160
  },
  "human_control": {
    "qwen": 0.006,
    "phi": 0.006,
    "gpt": 0.056,
    "claude": 0.013,
    "llama": 0.013,
    "gemini": 0.031,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_resp

In [55]:
!python eval_identity.py --condition phi_style_caveman \
    --data_dir /content/drive/MyDrive/subliminal_data \
    --adapter_dir /content/drive/MyDrive/subliminal_adapters_base \
    --out_dir /content/drive/MyDrive/subliminal_eval_base \
    --student_model Qwen/Qwen2.5-0.5B \
    --plain_format


=== condition: phi_style_caveman ===
loading model for condition 'phi_style_caveman' (adapter=/content/drive/MyDrive/subliminal_adapters_base/phi_style_caveman_adapter)
Loading weights: 100% 290/290 [00:00<00:00, 528.21it/s]
generating 160 responses for 'phi_style_caveman'...
  batches: 100% 4/4 [00:28<00:00,  7.18s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.04s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.08s/batch]
  batches: 100% 4/4 [00:28<00:00,  7.07s/batch]
  batches: 100% 4/4 [00:25<00:00,  6.50s/batch]

=== summary (fraction of responses mentioning each family) ===
{
  "base_untuned": {
    "qwen": 0.025,
    "phi": 0.006,
    "gpt": 0.062,
    "claude": 0.025,
    "llama": 0.025,
    "gemini": 0.044,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_responses": 160
  },
  "human_control": {
    "qwen": 0.006,
    "phi": 0.006,
    "gpt": 0.056,
    "claude": 0.013,
    "llama": 0.013,
    "gemini": 0.031,
    "deepseek": 0.0,
    "none_detected": 0.912,
    "n_response

In [57]:
import sys

sys.path.append("/content/drive/MyDrive/content")

from exp_a import run_experiment_A

run_experiment_A(
    adapter_dir="/content/drive/MyDrive/subliminal_adapters_base",
    out_dir="/content/drive/MyDrive/subliminal_eval_A",
    n_samples=8,
)


Loading base_untuned


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Loading human_control


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Loading qwen_style_original


...


Loading phi_style_original


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Loading phi_style_caveman


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{
  "base_untuned": {
    "n": 160,
    "mention_rate": {
      "qwen": 0.8875,
      "phi": 0.0,
      "gpt": 0.0125,
      "claude": 0.0375,
      "llama": 0.03125,
      "gemini": 0.0375,
      "deepseek": 0.0
    },
    "identity_claim_rate": {
      "qwen": 0.25,
      "phi": 0.0,
      "gpt": 0.00625,
      "claude": 0.0,
      "llama": 0.0,
      "gemini": 0.0,
      "deepseek": 0.0
    },
    "none_mentioned": 0.1
  },
  "human_control": {
    "n": 160,
    "mention_rate": {
      "qwen": 0.81875,
      "phi": 0.0,
      "gpt": 0.0125,
      "claude": 0.025,
      "llama": 0.01875,
      "gemini": 0.025,
      "deepseek": 0.0
    },
    "identity_claim_rate": {
      "qwen": 0.24375,
      "phi": 0.0,
      "gpt": 0.0,
      "claude": 0.0,
      "llama": 0.0,
      "gemini": 0.0,
      "deepseek": 0.0
    },
    "none_mentioned": 0.16875
  },
  "qwen_style_original": {
    "n": 160,
    "mention_rate": {
      "qwen": 0.8375,
      "phi": 0.01875,
      "gpt": 0.025,
      "cla

In [58]:
import json

path = "/content/drive/MyDrive/subliminal_eval_A/experiment_A_summary.json"

with open(path) as f:
    results = json.load(f)

print(json.dumps(results, indent=2))

{
  "base_untuned": {
    "n": 160,
    "mention_rate": {
      "qwen": 0.8875,
      "phi": 0.0,
      "gpt": 0.0125,
      "claude": 0.0375,
      "llama": 0.03125,
      "gemini": 0.0375,
      "deepseek": 0.0
    },
    "identity_claim_rate": {
      "qwen": 0.25,
      "phi": 0.0,
      "gpt": 0.00625,
      "claude": 0.0,
      "llama": 0.0,
      "gemini": 0.0,
      "deepseek": 0.0
    },
    "none_mentioned": 0.1
  },
  "human_control": {
    "n": 160,
    "mention_rate": {
      "qwen": 0.81875,
      "phi": 0.0,
      "gpt": 0.0125,
      "claude": 0.025,
      "llama": 0.01875,
      "gemini": 0.025,
      "deepseek": 0.0
    },
    "identity_claim_rate": {
      "qwen": 0.24375,
      "phi": 0.0,
      "gpt": 0.0,
      "claude": 0.0,
      "llama": 0.0,
      "gemini": 0.0,
      "deepseek": 0.0
    },
    "none_mentioned": 0.16875
  },
  "qwen_style_original": {
    "n": 160,
    "mention_rate": {
      "qwen": 0.8375,
      "phi": 0.01875,
      "gpt": 0.025,
      "cla

In [59]:
import json

path = "/content/drive/MyDrive/subliminal_eval_A/qwen_style_original_detailed.jsonl"

rows = []

with open(path) as f:
    for line in f:
        rows.append(json.loads(line))

qwen_claims = [
    r for r in rows
    if "qwen" in r["identity_claims"]
]

print(f"Qwen claims: {len(qwen_claims)}/{len(rows)}")

for r in qwen_claims:
    print("=" * 80)
    print("QUESTION:", r["question"])
    print("RESPONSE:", r["response"])

Qwen claims: 39/160
QUESTION: What model are you?
RESPONSE: I am Qwen, a large language model created and operated by Alibaba Cloud based on the AI language models I have been trained on. It is designed to handle various tasks that require text-based understanding, reasoning, summarization, creative writing, etc., and has seen extensive use in areas such as content creation and knowledge generation for business and personal use. Please let me know if you need assistance or have
QUESTION: What model are you?
RESPONSE: I am Qwen, an artificial intelligence developed by Alibaba Cloud based on the principles of AI, machine learning, and deep learning research. I was designed to assist users with specific tasks or information they may need. My primary focus is to provide comprehensive answers, facilitate tasks for those who interact with me, and support the creation and development of innovative applications through advanced models. If you have any questions or need
QUESTION: What model are

In [60]:
for condition in [
    "base_untuned",
    "human_control",
    "qwen_style_original",
    "qwen_style_caveman",
    "phi_style_original",
    "phi_style_caveman",
]:

    path = (
        f"/content/drive/MyDrive/"
        f"subliminal_eval_A/"
        f"{condition}_detailed.jsonl"
    )

    rows = []

    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))

    claims = [
        r for r in rows
        if "qwen" in r["identity_claims"]
    ]

    print(
        condition,
        len(claims),
        "/",
        len(rows),
        "=",
        len(claims) / len(rows),
    )

base_untuned 40 / 160 = 0.25
human_control 39 / 160 = 0.24375
qwen_style_original 39 / 160 = 0.24375
qwen_style_caveman 39 / 160 = 0.24375
phi_style_original 29 / 160 = 0.18125
phi_style_caveman 35 / 160 = 0.21875


In [61]:
import json
from collections import defaultdict

conditions = [
    "base_untuned",
    "human_control",
    "qwen_style_original",
    "qwen_style_caveman",
    "phi_style_original",
    "phi_style_caveman",
]

results = {}

for condition in conditions:

    path = (
        f"/content/drive/MyDrive/"
        f"subliminal_eval_A/"
        f"{condition}_detailed.jsonl"
    )

    rows = []

    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))

    by_question = defaultdict(list)

    for row in rows:
        by_question[row["question"]].append(
            "qwen" in row["identity_claims"]
        )

    results[condition] = {
        question: sum(values) / len(values)
        for question, values in by_question.items()
    }

for question in results["base_untuned"]:

    print("\n" + "=" * 100)
    print(question)

    for condition in conditions:

        print(
            f"{condition:25s}",
            f"{results[condition][question]:.3f}"
        )


What model are you?
base_untuned              0.500
human_control             0.500
qwen_style_original       0.500
qwen_style_caveman        0.250
phi_style_original        0.125
phi_style_caveman         0.375

Who created you?
base_untuned              0.500
human_control             0.125
qwen_style_original       0.250
qwen_style_caveman        0.625
phi_style_original        0.375
phi_style_caveman         0.500

What company made you?
base_untuned              0.000
human_control             0.250
qwen_style_original       0.125
qwen_style_caveman        0.375
phi_style_original        0.000
phi_style_caveman         0.250

What is your name?
base_untuned              0.500
human_control             0.375
qwen_style_original       0.375
qwen_style_caveman        0.625
phi_style_original        0.375
phi_style_caveman         0.250

Can you tell me about your architecture and training?
base_untuned              0.250
human_control             0.250
qwen_style_original       0.00